# Quantum Computing: QAOA (Quantum Approximate Optimization Algorithm)

## Solving Combinatorial Problems on NISQ Devices

**Seminal Work**: Farhi, Goldstone, Gutmann (2014)  
**Problem**: Max-Cut (NP-hard) → partition graph vertices to maximize edge cuts  
**Quantum Advantage**: Even shallow circuits beat randomized algorithms


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

np.random.seed(42)

## Part 1: QAOA Theory

### Max-Cut Problem
Given graph G=(V,E), find partition x∈{0,1}^n to maximize:
```
C(x) = ∑_{(i,j)∈E} (1 - x_i x_j)
```

### QAOA Ansatz
```
H_P = ∑_{(i,j)} (1 - Z_i Z_j)/2    [problem Hamiltonian]
H_M = ∑_i X_i                      [mixer Hamiltonian]

U(β,γ) = ∏_p e^{-iβ_p H_M} e^{-iγ_p H_P}
```

### Approximation Guarantee
p=1 (depth 1): 0.692-approximation (vs 0.5 random, 0.878 classical Goemans-Williamson)


In [ ]:
class QAOASimulator:
    """QAOA for Max-Cut problem."""
    
    def __init__(self, edges, n_nodes):
        """Initialize QAOA.
        
        edges: list of (i,j) tuples
        n_nodes: number of nodes
        """
        self.edges = edges
        self.n = n_nodes
    
    def cut_value(self, x):
        """Compute cut value for assignment x."""
        cut = 0
        for i, j in self.edges:
            if x[i] != x[j]:
                cut += 1
        return cut
    
    def energy_qaoa(self, beta, gamma):
        """Expected cut value from QAOA circuit (mock)."""
        # In real QAOA, this is measured on quantum device
        # Mock: energy depends on angles
        n_edges = len(self.edges)
        energy = n_edges * (0.5 + 0.3*np.sin(gamma) * np.cos(beta))
        return energy
    
    def optimize_qaoa(self, p=1):
        """Optimize QAOA angles."""
        def objective(params):
            # params: [β_1, γ_1, β_2, γ_2, ...]
            energy = 0
            for i in range(p):
                energy -= self.energy_qaoa(params[2*i], params[2*i+1])
            return energy
        
        x0 = np.random.rand(2*p) * np.pi
        result = minimize(objective, x0, method='COBYLA')
        
        return result

# Create sample graph
edges = [(0, 1), (1, 2), (2, 3), (3, 0), (0, 2)]  # Pentagon with diagonal
qaoa = QAOASimulator(edges, n_nodes=4)

print("QAOA FOR MAX-CUT")
print("="*60)
print(f"Number of nodes: {qaoa.n}")
print(f"Number of edges: {len(qaoa.edges)}")
print(f"Problem: partition vertices to maximize edge cuts")

# Brute force optimal
best_cut = 0
for assignment in range(2**qaoa.n):
    x = [(assignment >> i) & 1 for i in range(qaoa.n)]
    cut = qaoa.cut_value(x)
    best_cut = max(best_cut, cut)

print(f"\nOptimal Max-Cut: {best_cut}")
print(f"Random assignment (expected): {len(edges)/2:.1f}")
print(f"QAOA p=1 (guaranteed): {best_cut * 0.692:.1f}")

## Part 2: Depth vs Quality Tradeoff


In [ ]:
# Simulate QAOA performance vs depth
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: Approximation ratio vs p
ax = axes[0]
p_values = np.arange(1, 11)
approx_ratios = [0.692 + 0.05*p - 0.002*p**2 for p in p_values]  # Mock convergence
approx_ratios = np.minimum(appr ox_ratios, 0.878)  # Bound by classical

ax.plot(p_values, approx_ratios, 'o-', linewidth=2, markersize=8, color='steelblue', label='QAOA')
ax.axhline(y=0.5, color='red', linestyle='--', linewidth=2, label='Random (50%)')
ax.axhline(y=0.878, color='green', linestyle='--', linewidth=2, label='Goemans-Williamson (87.8%)')
ax.set_xlabel('QAOA Depth (p)')
ax.set_ylabel('Approximation Ratio')
ax.set_title('QAOA: Quality vs Circuit Depth')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_ylim([0.4, 0.95])

# Plot 2: Gate count vs p
ax = axes[1]
gate_count = 2 * p_values * 10  # Roughly: 2p layers × 10 gates per layer
ax.plot(p_values, gate_count, 'o-', linewidth=2, markersize=8, color='darkgreen')
ax.axhline(y=100, color='orange', linestyle='--', linewidth=2, label='Typical NISQ limit')
ax.set_xlabel('QAOA Depth (p)')
ax.set_ylabel('Circuit Depth (gates)')
ax.set_title('Quantum Cost vs Circuit Depth')
ax.legend()
ax.grid(True, alpha=0.3)
ax.fill_between(p_values, 0, 100, alpha=0.1, color='green', label='Feasible region')

plt.tight_layout()
plt.savefig('SECTION_4_QUANTUM/qaoa.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nQAOA DEPTH VS PERFORMANCE")
print("="*60)
print(f"{'Depth (p)':>10} {'Approx Ratio':>15} {'Gates':>10} {'Feasible':>10}")
print("-"*60)
for p, ratio, gates in zip(p_values[:6], approx_ratios[:6], gate_count[:6]):
    feasible = "✓" if gates <= 100 else "✗"
    print(f"{p:>10} {ratio:>15.3f} {gates:>10.0f} {feasible:>10}")

## Key Insights

1. **Quantum speedup**: Even p=1 beats random assignment (69.2% vs 50%)
2. **Depth tradeoff**: Deeper circuits give better approximations but suffer from noise
3. **NISQ limitation**: Current hardware limits p to ~3-5 due to decoherence
4. **Practical use**: Portfolio optimization, maximum clique, graph coloring
5. **Research frontier**: Improving approximation with hardware-efficient angles

### References
- Farhi, E., Goldstone, J., & Gutmann, S. (2014). "A Quantum Approximate Optimization Algorithm"
